In [2]:
# Smart People Counting System

from ultralytics import YOLO
import cv2
from PIL import Image
from IPython.display import display, Video
import os
import csv
from pathlib import Path
from ipywidgets import FloatSlider, IntProgress

# Load YOLOv8 Model
model = YOLO("yolov8n.pt")

input_path = "people.mp4"

# IMAGE PROCESSING
if input_path.lower().endswith((".jpg", ".jpeg", ".png")):

    results = model(input_path)

    result = results[0]

    people_count = 0

    for box in result.boxes:
        if int(box.cls[0]) == 0:
            people_count += 1

    processed = result.plot()

    output_image = "processed_image.jpg"

    cv2.imwrite(
        output_image,
        cv2.cvtColor(processed, cv2.COLOR_RGB2BGR)
    )

    print("=" * 40)
    print("People Detected :", people_count)
    print("Saved As :", output_image)
    print("=" * 40)

    display(Image.open(output_image))

# VIDEO PROCESSING
else:

    cap = cv2.VideoCapture(input_path)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    output_video = "processed_video.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = cv2.VideoWriter(
        output_video,
        fourcc,
        fps,
        (width, height)
    )

    max_people = 0

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False
        )

        people_count = 0

        if results[0].boxes is not None:

            for box in results[0].boxes:

                if int(box.cls[0]) != 0:
                    continue

                people_count += 1

                x1, y1, x2, y2 = map(int, box.xyxy[0])

                conf = float(box.conf[0])

                if box.id is not None:
                    track_id = int(box.id[0])
                else:
                    track_id = -1

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    frame,
                    f"ID:{track_id} {conf:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

        max_people = max(max_people, people_count)

        cv2.putText(
            frame,
            f"People: {people_count}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

        cv2.putText(
            frame,
            f"Maximum: {max_people}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 0, 0),
            2
        )

        out.write(frame)

    cap.release()
    out.release()

    print("=" * 45)
    print("Video Processing Completed")
    print("Maximum People Count :", max_people)
    print("Output Saved As :", output_video)
    print("=" * 45)

    Video(output_video, embed=True)

Video Processing Completed
Maximum People Count : 18
Output Saved As : processed_video.mp4


In [1]:
confidence = FloatSlider(
    value=0.5,
    min=0.1,
    max=1.0,
    step=0.05,
    description='Confidence'
)

display(confidence)

NameError: name 'FloatSlider' is not defined